[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/lab10_ai_analyst_agent.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Lab 10 — <your last name>` (click the title at the top left).

**To submit:** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**API key.** In Colab this lab runs in **replay mode** (recorded examples) unless you add a key. For live mode, ask the instructor for one and add it in **Secrets** (key icon, left sidebar) as `ANTHROPIC_API_KEY`, with **Notebook access** on. Never paste a key into a cell.

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Lab 10 — AI analysts: tool use over your data
*Week 10. Baseline: sections 1–5. Powerups: section 6.*

An AI analyst is a language model with a tool. You ask a question in English; the model writes SQL, the
notebook runs it, the model reads the rows and answers. This week you build that loop from about 100 lines
of Python, point it at a heart-failure readmissions table built from live CMS data, and then do the part
that matters: check every answer with your own SQL.

**How labs work.** Save a copy in Drive, work in your copy, share it back with the instructor
when you submit. Every lab has a **baseline** (everyone finishes it in the session) and
**powerups** (optional depth). AI assistants may write code for you; **you own every line** and
may be asked to explain any of it. Cells marked `# TODO` are yours to fill in.

**No API key? The lab still runs.** In Colab the agent runs in **replay mode** unless you add a key (section 3): it replays recorded example transcripts, clearly labelled. The SQL in them runs for real; the
words are a recording. Every baseline step works in either mode.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


In [ ]:
# --- Week 10 packages: anthropic ------------------------------------------------
# Not every base image has them (Hex does not). Installs once per kernel session.
import importlib.util, shutil, subprocess, sys

_need = [pkg for pkg, mod in [("anthropic", "anthropic")]
         if importlib.util.find_spec(mod) is None]
if _need:
    if shutil.which("uv"):  # Hex: the kernel's virtualenv has no pip module, but uv is on the PATH
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_need], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=True)
    importlib.invalidate_caches()
print("week 10 packages ready")


## 1. The table the agent will see
The cell below builds `hospitals`: one row per hospital in Medicare's heart-failure readmissions program
(HRRP), with hospital details and the Care Compare heart-failure readmission rate attached. We use heart
failure because it has numbers; the hospital-wide measure (`Hybrid_HWR`) has none in the current release.

The agent never sees this DataFrame directly. It sees `SCHEMA`, a paragraph of text describing the table
(next cell), and whatever rows its queries return.

**Baseline:** report the number of rows and how many have an `excess_ratio`. Then check two claims that
`SCHEMA` makes, with code: (a) `facility_id` is unique and 6 characters, and (b) `penalized` is missing
exactly when `excess_ratio` is missing. If `SCHEMA` lies, the agent will repeat the lie.

In [ ]:
# Three CMS files, read as text, turned into one tidy table the agent can query.
hrrp = load_cms("hrrp")
hgi = load_cms("hospital_general_info")
uhv = load_cms("unplanned_visits")

hf = hrrp[hrrp["Measure Name"].eq("READM-30-HF-HRRP")]
hospitals = pd.DataFrame({
    "facility_id":     hf["Facility ID"],
    "facility_name":   hf["Facility Name"],
    "state":           hf["State"],
    "hf_discharges":   to_num(hf["Number of Discharges"]),
    "hf_readmissions": to_num(hf["Number of Readmissions"]),   # 'Too Few to Report' -> missing, not 0
    "excess_ratio":    to_num(hf["Excess Readmission Ratio"]),
    "predicted_rate":  to_num(hf["Predicted Readmission Rate"]),
    "expected_rate":   to_num(hf["Expected Readmission Rate"]),
})
info = hgi[["Facility ID", "City/Town", "County/Parish", "Hospital Type", "Hospital Ownership",
            "Emergency Services", "Hospital overall rating"]].copy()
info.columns = ["facility_id", "city", "county", "hospital_type", "ownership", "emergency_services", "star_rating"]
hospitals = hospitals.merge(info, on="facility_id", how="left")
hospitals["star_rating"] = to_num(hospitals["star_rating"])

rate = uhv[uhv["Measure ID"].eq("READM_30_HF")]
rate = pd.DataFrame({"facility_id": rate["Facility ID"], "hf_readm_rate": to_num(rate["Score"])})
hospitals = hospitals.merge(rate, on="facility_id", how="left")

# 1 = in the penalty range, 0 = not, missing = no ratio published (never 0).
hospitals["penalized"] = (hospitals["excess_ratio"] > 1).astype(float).where(hospitals["excess_ratio"].notna())
hospitals = hospitals.reset_index(drop=True)
print(hospitals.shape, "|", int(hospitals["excess_ratio"].notna().sum()), "hospitals with an excess readmission ratio")


In [ ]:
import re

# What the agent is told about the table. It cannot see the data until it queries it,
# so this text is all it knows. Every claim in it should be true; you check that in section 1.
SCHEMA = """Table: hospitals. One row per hospital in Medicare's Hospital Readmissions Reduction Program
(HRRP) for heart failure, discharges July 2021 - June 2024; about 3,000 rows, all acute care hospitals.
Missing values are NULL.

facility_id         VARCHAR  CMS certification number (6 characters; text, keep leading zeros)
facility_name       VARCHAR  hospital name, upper case
state               VARCHAR  2-letter postal code (includes DC and territories such as PR)
city, county        VARCHAR  from Hospital General Information (NULL for ~20 hospitals not in that file)
hospital_type       VARCHAR  from Hospital General Information
ownership           VARCHAR  e.g. 'Proprietary', 'Voluntary non-profit - Private',
                             'Voluntary non-profit - Church', 'Government - Local'
emergency_services  VARCHAR  'Yes' or 'No'
star_rating         DOUBLE   CMS overall star rating 1-5; NULL if not rated
hf_discharges       DOUBLE   heart-failure discharges in the HRRP period; NULL if not reported
hf_readmissions     DOUBLE   heart-failure readmissions; NULL if 'Too Few to Report' or not available
excess_ratio        DOUBLE   HRRP excess readmission ratio = predicted / expected; NULL if not published
predicted_rate      DOUBLE   predicted readmission rate, percent
expected_rate       DOUBLE   expected readmission rate, percent
hf_readm_rate       DOUBLE   risk-standardized 30-day heart-failure readmission rate, percent
                             (Care Compare, July 2023 - June 2025)
penalized           DOUBLE   1 if excess_ratio > 1 (the HRRP penalty range), 0 if not, NULL if no ratio

Not in this table: hospital-wide readmission rates, other conditions, urban/rural flags,
patient-level data, payments or penalty amounts."""

QUERY_TOOL = {
    "name": "query_hospitals",
    "description": ("Run one read-only DuckDB SELECT against the `hospitals` table and get the rows back "
                    "as JSON (at most 100 rows). Use it for every number you report."),
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {"sql": {"type": "string", "description": "A single SELECT (or WITH ... SELECT) statement."}},
        "required": ["sql"],
        "additionalProperties": False,
    },
}

# The tool gets its own DuckDB connection that can see ONE table and no files.
# (The course sql() helper can see every DataFrame in the notebook; the agent should not.)
_agent_db = duckdb.connect(config={"enable_external_access": False})
_agent_db.register("hospitals", hospitals)
_FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|alter|create|attach|detach|copy|pragma|install|load|export|import|call|set)\b", re.I)

def query_hospitals(sql: str, max_rows: int = 100) -> dict:
    """The agent's only tool: run a SELECT, return rows as plain Python. Refuses anything else."""
    q = sql.strip().rstrip(";").strip()
    if ";" in q:
        raise ValueError("one statement at a time")
    if not re.match(r"(?is)^(select|with)\b", q):
        raise ValueError("only SELECT queries are allowed")
    if _FORBIDDEN.search(q):
        raise ValueError(f"keyword not allowed: {_FORBIDDEN.search(q).group(0)}")
    df = _agent_db.execute(q).df()
    return {"row_count": len(df), "truncated": len(df) > max_rows, "columns": list(df.columns),
            "rows": json.loads(df.head(max_rows).to_json(orient="records"))}


In [ ]:
# TODO: (a) is facility_id unique, and always 6 characters?
# TODO: (b) is penalized missing exactly when excess_ratio is missing?


**Rows, rows with a ratio, and the two schema checks:** *(edit this cell)*

## 2. The tool
`query_hospitals(sql)` is the agent's only way to touch data. Read it: it runs one `SELECT` on a
separate DuckDB connection that can see the `hospitals` table and nothing else (no files, no other
DataFrames), refuses anything that writes, and returns at most 100 rows as JSON. `QUERY_TOOL` is the
description the model reads to decide when and how to call it.

**Baseline:** call the tool yourself three times: one query that works, one with a SQL mistake, and one
that should be refused (for example `DROP TABLE hospitals`). Report what each returned. One sentence:
why does the tool get its own connection instead of using the course `sql()` helper?

In [ ]:
# TODO: three calls to query_hospitals(...): one that works, one with a SQL error, one that is refused.
query_hospitals("SELECT state, COUNT(*) AS n FROM hospitals GROUP BY state ORDER BY n DESC LIMIT 5")


**What the three calls returned, and why a separate connection:** *(edit this cell)*

## 3. The agent loop
**The API key (Colab).** In Colab this lab runs in **replay mode** by default, and every baseline step works that way. For **live mode**, ask the instructor for a key. Then open **Secrets** (the key icon in the left sidebar) → **Add new secret**, name it `ANTHROPIC_API_KEY`, paste the key as the value, and switch on **Notebook access**. If Colab asks whether this notebook may read the secret, allow it. Secrets are not saved in the notebook and do not travel when you share it. Never put a key in a cell, a screenshot, or a commit. The cell below looks for a key in this order: a Hex secret, Colab Secrets, an environment variable. If it prints `MODE = REPLAY`, it found none and the agent replays recordings.

**The loop** (`run_agent`, next cell) is the whole idea of an agent:
1. Send the question, the system prompt (with `SCHEMA`), and the tool description to the model.
2. If the model answers with a **tool call**, run `query_hospitals` with its SQL and send the rows back.
3. Repeat until the model stops asking for tools. Its last text is the answer.

Every call reports **tokens used**; `run_cost` turns them into dollars at list price. A question usually
takes two or three model calls and costs a few cents.

**Baseline:** read `run_agent` and `show` and answer in two sentences: where in the code does the model's SQL
actually run, and what does the model see when its SQL fails?

In [ ]:
import os, time, textwrap
import anthropic

MODEL = "claude-opus-5-5"
# US dollars per million tokens for this model (Anthropic list price, October 2026).
PRICE_PER_MTOK = {"input": 4.00, "output": 20.00, "cache_read": 0.20, "cache_write": 5.00}
MAX_TURNS = 8   # model calls per question; a runaway loop stops here

# Where the key comes from, tried in order:
#   1. Hex: a workspace secret named ANTHROPIC_API_KEY appears as a Python variable of that name.
#   2. Colab: the Secrets panel (key icon, left sidebar), secret ANTHROPIC_API_KEY, notebook access on.
#   3. Anywhere else (a laptop): an environment variable.
# No key found -> replay mode. In Colab that is the default.
_key = globals().get("ANTHROPIC_API_KEY")
if not (isinstance(_key, str) and _key.strip()):
    try:
        from google.colab import userdata
        _key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:   # not in Colab, secret not set, or notebook access not granted
        _key = None
if not (isinstance(_key, str) and _key.strip()):
    _key = os.environ.get("ANTHROPIC_API_KEY")
API_KEY = _key.strip() if isinstance(_key, str) and _key.strip() else None
MODE = "live" if API_KEY else "replay"
client = anthropic.Anthropic(api_key=API_KEY) if API_KEY else None
print(f"MODE = {MODE.upper()}  (model {MODEL}, anthropic SDK {anthropic.__version__})")
if MODE == "replay":
    print("No ANTHROPIC_API_KEY found, so the agent will REPLAY recorded example transcripts. "
          "The SQL in them runs for real; the agent's words are a recording, not a live model.")


In [ ]:
SYSTEM = f"""You are a careful healthcare data analyst. Answer questions about U.S. hospitals by calling
the query_hospitals tool; never state a number you did not get from a query. Say which filter and which
denominator you used. If the table cannot answer the question, say so plainly and offer the closest thing
it can answer. Keep the final answer short.

{SCHEMA}"""

def run_agent(question: str, system: str = SYSTEM) -> dict:
    """Ask the model; run every tool call it makes; repeat until it stops asking for tools."""
    run = {"question": question, "mode": "live", "model": MODEL, "events": [], "usage": [], "answer": "", "stop": None}
    messages = [{"role": "user", "content": question}]
    for _ in range(MAX_TURNS):
        t0 = time.perf_counter()
        try:
            response = client.beta.messages.create(
                model=MODEL, max_tokens=16000, system=system, tools=[QUERY_TOOL], messages=messages,
                output_config={"effort": "medium"},
                # If a safety filter declines a request, retry it on Anthropic's recommended fallback model.
                betas=["server-side-fallback-2026-07-01"], fallbacks="default",
            )
        except anthropic.AuthenticationError:
            run["events"].append({"kind": "error", "text": "API key rejected. Ask the instructor."}); break
        except anthropic.RateLimitError:
            run["events"].append({"kind": "error", "text": "Rate limited. Wait a minute and rerun."}); break
        except anthropic.APIStatusError as e:
            run["events"].append({"kind": "error", "text": f"API error {e.status_code}: {e.message}"}); break
        except anthropic.APIConnectionError:
            run["events"].append({"kind": "error", "text": "Could not reach the API."}); break
        u = response.usage
        run["usage"].append({"input": u.input_tokens, "output": u.output_tokens,
                             "cache_read": u.cache_read_input_tokens or 0,
                             "cache_write": u.cache_creation_input_tokens or 0,
                             "seconds": round(time.perf_counter() - t0, 1)})
        run["stop"] = response.stop_reason
        # Send the assistant turn back exactly as received (it includes hidden thinking blocks).
        messages.append({"role": "assistant", "content": response.content})
        for block in response.content:
            if block.type == "text" and block.text.strip():
                run["events"].append({"kind": "text", "text": block.text})
        if response.stop_reason == "refusal":
            run["events"].append({"kind": "error", "text": "The model declined this request."}); break
        if response.stop_reason != "tool_use":
            break
        results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            sql_text = block.input.get("sql", "")
            run["events"].append({"kind": "tool_call", "tool": block.name, "sql": sql_text})
            try:
                if block.name != "query_hospitals":
                    raise ValueError(f"unknown tool {block.name}")
                out = query_hospitals(sql_text)
                content, err = json.dumps(out, default=str), None
            except Exception as e:  # the error goes back to the model, which can fix its SQL
                out, err = None, f"{type(e).__name__}: {e}"
                content = "ERROR: " + err
            run["events"].append({"kind": "tool_result", "result": out, "error": err})
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": content, "is_error": err is not None})
        messages.append({"role": "user", "content": results})   # all results in one message
    texts = [e["text"] for e in run["events"] if e["kind"] == "text"]
    run["answer"] = texts[-1] if texts else ""
    return run

def run_cost(run: dict) -> float:
    """Dollars for one run, from the token counts the API reported."""
    return sum(u[k] * PRICE_PER_MTOK[k] for u in run["usage"] for k in PRICE_PER_MTOK) / 1e6


In [ ]:
def show(run: dict, max_rows: int = 8):
    """Print a run as a transcript: what the agent said, each tool call, and what came back."""
    print("=" * 96)
    tag = "LIVE" if run["mode"] == "live" else "REPLAY (recorded example, not a live model)"
    print(f"[{tag}]  Q: {run['question']}")
    for e in run["events"]:
        if e["kind"] == "text":
            print("\nAGENT: " + e["text"].strip())
        elif e["kind"] == "tool_call":
            print(f"\n  -> {e['tool']}(sql=)\n" + textwrap.indent(e["sql"].strip(), "     "))
        elif e["kind"] == "tool_result":
            if e["error"]:
                print("  <- ERROR " + e["error"])
            else:
                r = e["result"]
                print(f"  <- {r['row_count']} row(s)" + (" (truncated)" if r.get("truncated") else ""))
                if r["rows"]:
                    print(textwrap.indent(pd.DataFrame(r["rows"]).head(max_rows).to_string(index=False), "     "))
        elif e["kind"] == "error":
            print("\n!! " + e["text"])
    if run["usage"]:
        tin = sum(u["input"] for u in run["usage"]); tout = sum(u["output"] for u in run["usage"])
        print(f"\n[{len(run['usage'])} model call(s), {tin:,} input + {tout:,} output tokens, ${run_cost(run):.4f}]")
    else:
        print("\n[replay: no model calls, $0]")


In [ ]:
# Recorded example transcripts for REPLAY mode (written by course staff, 2026-10-26).
# They show the shape of a real run: each step is the SQL a tool-using analyst would send, and that SQL
# runs for real against your table. The agent's sentences are a recording, not a live model, and the
# numbers in them were true on 2026-10-26. Two of the five answers contain mistakes of the kind live
# agents make. Finding them is part of section 5.
QUESTIONS = [
    "How many hospitals have a heart-failure excess readmission ratio, and what share of them are in the penalty range?",
    "What share of Florida hospitals are in the heart-failure penalty range?",
    "Which Connecticut hospital has the highest heart-failure excess readmission ratio?",
    "Do for-profit hospitals have worse heart-failure readmission performance than non-profit hospitals?",
    "What is the average hospital-wide readmission rate in Massachusetts?",
]

RECORDED = {
    QUESTIONS[0]: {
        "steps": [("I'll count hospitals with a published ratio and average the penalized flag over those same hospitals.",
                   "SELECT COUNT(excess_ratio) AS n_with_ratio,\n       ROUND(100 * AVG(penalized), 1) AS pct_penalty_range\nFROM hospitals")],
        "answer": ("2,621 hospitals have a published heart-failure excess readmission ratio. Of those, 48.9% have a ratio "
                   "above 1, which puts them in the HRRP penalty range. The denominator is the 2,621 hospitals with a "
                   "ratio; the 434 rows with no ratio are excluded, not counted as 'not penalized'."),
    },
    QUESTIONS[1]: {
        "steps": [("I'll count Florida hospitals and how many have an excess ratio above 1.",
                   "SELECT COUNT(*) AS n_hospitals,\n       SUM(CASE WHEN excess_ratio > 1 THEN 1 ELSE 0 END) AS n_penalty_range,\n"
                   "       ROUND(100.0 * SUM(CASE WHEN excess_ratio > 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct\n"
                   "FROM hospitals\nWHERE state = 'FL'")],
        "answer": ("Florida has 169 hospitals in the table, and 108 of them have an excess readmission ratio above 1, "
                   "so 63.9% of Florida hospitals are in the heart-failure penalty range."),
    },
    QUESTIONS[2]: {
        "steps": [("I'll list Connecticut hospitals with a ratio, highest first.",
                   "SELECT facility_name, city, excess_ratio, hf_discharges\nFROM hospitals\n"
                   "WHERE state = 'CT' AND excess_ratio IS NOT NULL\nORDER BY excess_ratio DESC\nLIMIT 5")],
        "answer": ("Hartford Hospital (Hartford) has the highest ratio in Connecticut, 1.1395, on 1,305 heart-failure "
                   "discharges. Next are Norwalk Hospital (1.1177) and Day Kimball Hospital (1.1082). A ratio of 1.14 "
                   "means about 14% more readmissions than CMS expected for that hospital's patients."),
    },
    QUESTIONS[3]: {
        "steps": [("I'll compare the average excess ratio for proprietary and non-profit hospitals.",
                   "SELECT ownership, COUNT(*) AS n, ROUND(AVG(excess_ratio), 4) AS avg_excess_ratio\n"
                   "FROM hospitals\nWHERE ownership IN ('Proprietary', 'Voluntary non-profit - Private')\n"
                   "  AND excess_ratio IS NOT NULL\nGROUP BY ownership")],
        "answer": ("Yes. For-profit (proprietary) hospitals average an excess ratio of 1.0155 (n = 474), against "
                   "0.9962 for non-profit hospitals (n = 1,357). For-profit hospitals do about 2% worse than expected "
                   "on heart-failure readmissions."),
    },
    QUESTIONS[4]: {
        "steps": [("The table has no hospital-wide readmission rate, only heart failure. I'll check what it has for Massachusetts.",
                   "SELECT COUNT(*) AS n_hospitals, COUNT(hf_readm_rate) AS n_with_rate,\n"
                   "       ROUND(AVG(hf_readm_rate), 2) AS avg_hf_readm_rate\nFROM hospitals\nWHERE state = 'MA'")],
        "answer": ("I can't answer that from this table: it has no hospital-wide readmission rate. The closest thing it "
                   "has is heart failure. Across the 49 Massachusetts hospitals with a published rate, the average "
                   "risk-standardized 30-day heart-failure readmission rate is 22.01%. That is an unweighted average "
                   "of hospital rates, not a statewide patient rate."),
    },
}

def replay_agent(question: str) -> dict:
    """Replay a recorded transcript. The SQL runs now, against your table; the words are the recording."""
    run = {"question": question, "mode": "replay", "model": MODEL + " (recorded)", "events": [], "usage": [],
           "answer": "", "stop": "end_turn"}
    rec = RECORDED.get(question)
    if rec is None:
        run["events"].append({"kind": "error", "text": "No recording for this question. Replay mode only knows "
                              "the example questions; live mode needs an API key (set up in lab 10, section 3)."})
        return run
    for narration, q in rec["steps"]:
        run["events"].append({"kind": "text", "text": narration})
        run["events"].append({"kind": "tool_call", "tool": "query_hospitals", "sql": q})
        try:
            run["events"].append({"kind": "tool_result", "result": query_hospitals(q), "error": None})
        except Exception as e:
            run["events"].append({"kind": "tool_result", "result": None, "error": f"{type(e).__name__}: {e}"})
    run["events"].append({"kind": "text", "text": rec["answer"]})
    run["answer"] = rec["answer"]
    return run

def ask(question: str, **kw) -> dict:
    return run_agent(question, **kw) if MODE == "live" else replay_agent(question)


**Where the SQL runs, and what the model sees on failure:** *(edit this cell)*

## 4. Ask it five questions
The cell asks the five questions in `QUESTIONS` and prints each transcript: the agent's words, every SQL
call, and the rows that came back. In live mode you may swap any of them for your own questions (keep at
least one the table *cannot* answer).

**Baseline:** run it and read every transcript. Do not judge the answers yet; that is section 5.

In [ ]:
runs = [ask(q) for q in QUESTIONS]
for r in runs:
    show(r)


## 5. Check every answer yourself
An answer you have not checked is a rumour. For each question, write your **own** SQL with the course `sql()`
helper (it runs against the same `hospitals` DataFrame), compare, and fill in the table. Look past the number:
was the **denominator** right, the **filter** right, the **claim** supported?

**Baseline:** five checking queries, the table filled in, and the cost line below. Then two sentences: which
mistake would you have missed if you only read the agent's answer?

In [ ]:
# TODO: one checking query per question. The first is done for you.
sql("""SELECT COUNT(excess_ratio) AS n_with_ratio, ROUND(100 * AVG(penalized), 1) AS pct_penalty_range
       FROM hospitals""")


In [ ]:
# Tokens and dollars per question (all zeros in replay mode).
cost_table = pd.DataFrame([{
    "question": r["question"][:60],
    "model_calls": len(r["usage"]),
    "input_tokens": sum(u["input"] for u in r["usage"]),
    "output_tokens": sum(u["output"] for u in r["usage"]),
    "seconds": round(sum(u["seconds"] for u in r["usage"]), 1),
    "usd": round(run_cost(r), 4),
} for r in runs])
print(f"mode {MODE}: total ${cost_table['usd'].sum():.3f} for {len(runs)} questions")
cost_table


**Right or wrong (edit this cell):**

| # | Agent's answer (short) | Your check | Right / partly / wrong | What went wrong, if anything |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

**Cost:** *(total tokens and dollars for the five, or "replay mode, $0")*

**The mistake you would have missed:** *(edit this cell)*

## 6. Powerups
**(a) Build your own agent with a new tool.** Add a second tool and let the agent choose between them. Ideas:
`lookup_hospital(name)` that does a fuzzy name search and returns `facility_id`s (so "Yale" finds
YALE-NEW HAVEN HOSPITAL), or `compare_groups(column, metric)` that returns mean, sd and n per group so the
agent stops reporting bare means. You need three things: a Python function, a tool description dict like
`QUERY_TOOL`, and a branch in the dispatch inside `run_agent`. Show one transcript where the agent uses it.

**(b) Same questions, different effort.** Rerun the five with `output_config={"effort": "low"}` and then
`"high"`. Compare correctness, tokens and dollars. Is the cheaper run worse on these questions?

**(c) Prompt caching.** The system prompt and tool description are identical on every call. Read about
prompt caching in the Anthropic docs, add a `cache_control` breakpoint, and show `cache_read` tokens in the
cost table. (Caching only starts above a minimum prompt length; find out what it is.)

In [ ]:
# TODO (powerup a): a second tool.
# def lookup_hospital(name: str) -> dict: ...
# LOOKUP_TOOL = {"name": "lookup_hospital", "description": "...", "strict": True, "input_schema": {...}}
# then pass tools=[QUERY_TOOL, LOOKUP_TOOL] and dispatch on block.name inside run_agent.
